In [ ]:
def check_neighbors(df_history, df_upcoming, feature_cols, n_neighbors=5):
    X_hist = df_history[feature_cols].values
    X_up = df_upcoming.values

    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
    nn.fit(X_hist)                 
    distances, indices = nn.kneighbors(X_up)  

    rows = []
    for i, up_idx in enumerate(df_upcoming.index):
        for k in range(n_neighbors):
            hist_row = df_history.iloc[indices[i, k]]
            rows.append({
                "upcoming_index": up_idx,
                "neighbor_rank": k + 1,
                "history_index": hist_row.name,
                "distance": distances[i, k],
                **hist_row.to_dict()   # append all history columns
            })

    neighbors_df = pd.DataFrame(rows)
    return neighbors_df


def shap_for_statsmodels_logit(logit_model, X_background, X_explain, upcoming_fighters, nsamples=100):
    """
    Compute SHAP values for a statsmodels Logit model.

    Parameters
    ----------
    logit_model : statsmodels.discrete.discrete_model.BinaryResults
        Fitted Logit model (result of model.fit()).
    X_background : pd.DataFrame or np.ndarray
        Background dataset for SHAP (usually training set or a subsample).
    X_explain : pd.DataFrame or np.ndarray
        Samples to explain.
    nsamples : int
        Number of Monte Carlo samples for Kernel SHAP.

    Returns
    -------
    shap_values : np.ndarray
        SHAP values in log-odds space.
    explainer : shap.KernelExplainer
    """
    X_background_const = sm.add_constant(X_background.values, has_constant='add')  # only adds if missing
    X_explain_const = sm.add_constant(X_explain.values, has_constant='add')

    def predict_logit(X):
        # statsmodels returns probabilities by default
        # SHAP works better in log-odds for linear models
        p = logit_model.predict(X)
        eps = 1e-9
        return np.log(p / (1 - p + eps))

    explainer = shap.KernelExplainer(predict_logit, X_background_const)
    shap_values = explainer.shap_values(X_explain_const, nsamples=nsamples)

    sample = X_explain_const[0,:] # keep as 2D
    shap_values_single = shap_values[0]

    # Create a force plot

    # Assume shap_values_single (1D array) and X_explain (DataFrame) are ready
    feature_names = ['constant'] + list(X_explain.columns)
    shap_vals = shap_values_single

    # Compute probability
    logit = explainer.expected_value + shap_vals.sum()
    prob_shap = 1 / (1 + np.exp(-logit))

    # Model probability (direct from statsmodels)
    prob_model = logit_model.predict(sample)[0]

    return prob_model, shap_vals



def shap_pipeline(upcoming_df, df_history, feats_list, model_list, scaler_list):

    upcoming_fighters = upcoming_df[['fighter_red', 'fighter_blue']].values

    for model, feats, scaler in zip(model_list, feats_list, scaler_list):
        df_background = pd.DataFrame(
            scaler.transform(df_history[feats]),
            columns=feats
        )
    
        df_background = df_background.drop(columns=['open_red', 'open_blue'])
        nn = NearestNeighbors(n_neighbors=500, metric='euclidean')
        nn.fit(df_background)

        event_df = upcoming_df[feats+['fighter_red', 'fighter_blue']].dropna().reset_index(drop=True)

        fig, axes = plt.subplots(nrows=event_df.shape[0], figsize=(20,60))
        for i, row in event_df.iterrows(): 
            
            fight = row[feats]
            fight_df = pd.DataFrame([fight.values], columns=feats)

            df_explain = pd.DataFrame(
                scaler.transform(fight_df),
                columns=feats
            )

            df_explain = df_explain.drop(columns=['open_red', 'open_blue'])

            distances, indices = nn.kneighbors(df_explain.values.reshape(1, -1))
            nearest_fights = df_background.iloc[indices[0]]
            upcoming_fighters = row[['fighter_red', 'fighter_blue']].values
            fighter_red = upcoming_fighters[0]
            fighter_blue = upcoming_fighters[1]

            prob_model, shap_values  = shap_for_statsmodels_logit(model, nearest_fights, df_explain, upcoming_fighters)
            pred_winner = fighter_red if prob_model >= .5 else fighter_blue
            pred_color = 'red' if pred_winner == fighter_red else 'blue'
            pred_prob = prob_model if pred_color == 'red' else 1-prob_model

            feature_names = ['constant'] + list(df_explain.columns)
            colors = ['blue' if v < 0 else 'red' for v in shap_values]

            axes[i].barh(feature_names, shap_values, color=colors)
            axes[i].axvline(0, color='black', linewidth=0.8)
            axes[i].set_xlabel("SHAP value (log-odds)")
            axes[i].set_title(f'{fighter_red} vs {fighter_blue}, Pred Winner Color:{pred_color}, Fighter:{pred_winner}, Probability:{pred_prob:.2f}')

        plt.tight_layout()
        plt.show()